In [1]:
import sqlite3
import os

db_path = "whisky_auctions.db"

try:
    conn.close()
except:
    pass

if os.path.exists(db_path):
    os.remove(db_path)
    print("Removed existing database")

conn = sqlite3.connect(db_path)
cursor = conn.cursor()
print("Database created")

Removed existing database
Database created


In [2]:
schema = """
CREATE TABLE countries (
    id      INTEGER PRIMARY KEY,
    name    TEXT NOT NULL UNIQUE
);

CREATE TABLE distilleries (
    id          INTEGER PRIMARY KEY,
    name        TEXT NOT NULL UNIQUE,
    region      TEXT,
    country_id  INTEGER REFERENCES countries(id),
    status      TEXT CHECK(status IN ('Operational','Closed','Demolished','Unknown'))
);

CREATE TABLE bottlers (
    id          INTEGER PRIMARY KEY,
    name        TEXT NOT NULL UNIQUE,
    country_id  INTEGER REFERENCES countries(id),
    is_ob       BOOLEAN DEFAULT FALSE
);

CREATE TABLE cask_types (
    id    INTEGER PRIMARY KEY,
    name  TEXT NOT NULL UNIQUE
);

CREATE TABLE auctions (
    id              INTEGER PRIMARY KEY,
    auction_number  INTEGER NOT NULL UNIQUE,
    end_date        DATE,
    url             TEXT
);

CREATE TABLE lots (
    id              INTEGER PRIMARY KEY,
    lot_id          INTEGER NOT NULL UNIQUE,
    lot_number      TEXT,
    auction_id      INTEGER REFERENCES auctions(id),
    distillery_id   INTEGER REFERENCES distilleries(id),
    bottler_id      INTEGER REFERENCES bottlers(id),
    country_id      INTEGER REFERENCES countries(id),
    cask_type_id    INTEGER REFERENCES cask_types(id),
    title           TEXT NOT NULL,
    winning_bid     REAL,
    distilled_date  TEXT,
    bottled_date    TEXT,
    age_years       REAL,
    abv             REAL,
    volume_cl       INTEGER,
    bottle_number   INTEGER,
    total_bottles   INTEGER,
    cask_number     TEXT,
    is_ob           BOOLEAN DEFAULT FALSE,
    scraped_at      TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

for statement in schema.strip().split(";"):
    statement = statement.strip()
    if statement:
        cursor.execute(statement)

conn.commit()
print("Schema created successfully")

Schema created successfully


In [3]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
tables = cursor.fetchall()
print("Tables created:")
for table in tables:
    print(f"  {table[0]}")

Tables created:
  auctions
  bottlers
  cask_types
  countries
  distilleries
  lots


In [4]:
cursor.execute("PRAGMA table_info(lots)")
columns = cursor.fetchall()

print(f"{'col':>4}  {'name':<20}  {'type':<12}  {'nullable':<10}  {'default'}")
print("-" * 65)
for col in columns:
    cid, name, dtype, notnull, default, pk = col
    nullable = "NOT NULL" if notnull else "nullable"
    default = default or ""
    print(f"{cid:>4}  {name:<20}  {dtype:<12}  {nullable:<10}  {default}")

 col  name                  type          nullable    default
-----------------------------------------------------------------
   0  id                    INTEGER       nullable    
   1  lot_id                INTEGER       NOT NULL    
   2  lot_number            TEXT          nullable    
   3  auction_id            INTEGER       nullable    
   4  distillery_id         INTEGER       nullable    
   5  bottler_id            INTEGER       nullable    
   6  country_id            INTEGER       nullable    
   7  cask_type_id          INTEGER       nullable    
   8  title                 TEXT          NOT NULL    
   9  winning_bid           REAL          nullable    
  10  distilled_date        TEXT          nullable    
  11  bottled_date          TEXT          nullable    
  12  age_years             REAL          nullable    
  13  abv                   REAL          nullable    
  14  volume_cl             INTEGER       nullable    
  15  bottle_number         INTEGER       nulla

In [5]:
cursor.execute("""
    INSERT INTO countries (name) VALUES ('Scotland')
""")

cursor.execute("""
    INSERT INTO countries (name) VALUES ('USA')
""")

cursor.execute("""
    INSERT INTO bottlers (name, is_ob) VALUES ('American Single Cask', FALSE)
""")

cursor.execute("""
    INSERT INTO cask_types (name) VALUES ('Red Wine Barrel Finish')
""")

cursor.execute("""
    INSERT INTO auctions (auction_number, end_date, url)
    VALUES (177, '2026-03-08', 'https://www.scotchwhiskyauctions.com/auctions/226-the-177th-auction/')
""")

conn.commit()
print("Reference data inserted")

Reference data inserted


In [6]:
cursor.execute("""
    INSERT INTO lots (
        lot_id, lot_number, auction_id, bottler_id,
        country_id, cask_type_id, title, winning_bid,
        distilled_date, bottled_date, age_years, abv,
        volume_cl, bottle_number, total_bottles,
        cask_number, is_ob
    ) VALUES (
        867123, '177-05023',
        (SELECT id FROM auctions WHERE auction_number = 177),
        (SELECT id FROM bottlers WHERE name = 'American Single Cask'),
        (SELECT id FROM countries WHERE name = 'USA'),
        (SELECT id FROM cask_types WHERE name = 'Red Wine Barrel Finish'),
        '2Bar Spirits 2018 4 Year Old Bourbon Red Wine Barrel American Single Cask #0005 75cl',
        40.00,
        '10.31.2018', '05.01.2023', 4.0, 58.94,
        75, 36, 286, '0005', FALSE
    )
""")

conn.commit()
print("Lot inserted")

Lot inserted


In [7]:
cursor.execute("""
    SELECT 
        l.lot_id,
        l.lot_number,
        l.title,
        l.winning_bid,
        l.age_years,
        l.abv,
        b.name AS bottler,
        c.name AS country,
        ct.name AS cask_type,
        a.auction_number
    FROM lots l
    JOIN bottlers b  ON l.bottler_id  = b.id
    JOIN countries c ON l.country_id  = c.id
    JOIN cask_types ct ON l.cask_type_id = ct.id
    JOIN auctions a  ON l.auction_id   = a.id
""")

rows = cursor.fetchall()
for row in rows:
    print(row)

(867123, '177-05023', '2Bar Spirits 2018 4 Year Old Bourbon Red Wine Barrel American Single Cask #0005 75cl', 40.0, 4.0, 58.94, 'American Single Cask', 'USA', 'Red Wine Barrel Finish', 177)


In [8]:
cursor.execute("""
    SELECT 
        l.lot_id,
        l.lot_number,
        l.title,
        l.winning_bid,
        l.age_years,
        l.abv,
        b.name  AS bottler,
        c.name  AS country,
        ct.name AS cask_type,
        a.auction_number
    FROM lots l
    JOIN bottlers  b  ON l.bottler_id   = b.id
    JOIN countries c  ON l.country_id   = c.id
    JOIN cask_types ct ON l.cask_type_id = ct.id
    JOIN auctions  a  ON l.auction_id   = a.id
""")

columns = [desc[0] for desc in cursor.description]
rows = cursor.fetchall()

import pandas as pd
df = pd.DataFrame(rows, columns=columns)
print(df.T)

                                                                0
lot_id                                                     867123
lot_number                                              177-05023
title           2Bar Spirits 2018 4 Year Old Bourbon Red Wine ...
winning_bid                                                  40.0
age_years                                                     4.0
abv                                                         58.94
bottler                                      American Single Cask
country                                                       USA
cask_type                                  Red Wine Barrel Finish
auction_number                                                177


In [9]:
cursor.execute("SELECT lot_id, title, winning_bid FROM lots")
print(cursor.fetchall())

[(867123, '2Bar Spirits 2018 4 Year Old Bourbon Red Wine Barrel American Single Cask #0005 75cl', 40.0)]


In [10]:
import requests
from bs4 import BeautifulSoup
import time

def fetch_lot(lot_id, auction_slug="226-the-177th-auction"):
    url = f"https://www.scotchwhiskyauctions.com/auctions/{auction_slug}/{lot_id}/"
    response = requests.get(url, headers={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    })
    if response.status_code == 200:
        return BeautifulSoup(response.text, "html.parser")
    return None

soup = fetch_lot(867123)
print(f"Page fetched: {soup is not None}")
print(f"\nPage title: {soup.find('h1').text.strip()}")

Page fetched: True

Page title: 2Bar Spirits 2018 4 Year Old Bourbon Red Wine Barrel American Single Cask #0005 75cl


In [11]:
page_text = soup.get_text(separator="\n")
lines = [l.strip() for l in page_text.split("\n") if l.strip()]

for i, line in enumerate(lines):
    if any(word in line for word in [
        "Winning bid", "Lot number", "Distilled", 
        "Bottled", "Age", "Cask", "ABV", "Bottle Number"
    ]):
        print(f"Line {i:3d}: {line}")

Line   0: 2Bar Spirits 2018 4 Year Old Bourbon Red Wine Barrel American Single Cask #0005 75cl | The 177th Auction | Scotch Whisky Auctions
Line  17: 2Bar Spirits 2018 4 Year Old Bourbon Red Wine Barrel American Single Cask #0005 75cl
Line  22: Lot number: 177-05023
Line  23: Winning bid: £40
Line  24: American Single Cask is an independent bottler of American craft whiskeys. It started as an idea of combining their love of independent bottlers of Scotch with the boom in American craft whiskey distilling. They visit craft distilleries, pick single casks full of character and quality and bottle them to the same high standards set by the many great independent bottlers who came before them. Their aim is to provide the purest and most honest route to exploring the very best of American craft whiskey.
Line  30: Distilled: 10.31.2018
Line  31: Bottled: 05.01.2023
Line  32: Age: 4 Years Old
Line  33: Cask Type: Red Wine Barrel Finish
Line  34: Cask Number: 0005
Line  35: 58.94% ABV / 75cl
Li

In [16]:
def parse_lot(soup, lot_id):
    result = {
        "lot_id": lot_id,
        "title":         None,
        "lot_number":    None,
        "winning_bid":   None,
        "distilled_date": None,
        "bottled_date":  None,
        "age_years":     None,
        "cask_type":     None,
        "cask_number":   None,
        "abv":           None,
        "volume_cl":     None,
        "bottle_number": None,
        "total_bottles": None,
    }
    
    page_text = soup.get_text(separator="\n")
    lines = [l.strip() for l in page_text.split("\n") if l.strip()]
    
    h1 = soup.find("h1")
    result["title"] = h1.text.strip() if h1 else None
    
    for line in lines:
        if line.startswith("Lot number:"):
            result["lot_number"] = line.replace("Lot number:", "").strip()
            
        elif line.startswith("Winning bid:"):
            raw = line.replace("Winning bid:", "").strip()
            raw = raw.replace("£", "").replace(",", "").strip()
            try:
                result["winning_bid"] = float(raw)
            except ValueError:
                result["winning_bid"] = None
                
        elif line.startswith("Distilled:"):
            result["distilled_date"] = line.replace("Distilled:", "").strip()
            
        elif line.startswith("Bottled:"):
            result["bottled_date"] = line.replace("Bottled:", "").strip()
            
        elif line.startswith("Age:"):
            raw = line.replace("Age:", "").strip()
            import re
            match = re.search(r"(\d+\.?\d*)", raw)
            result["age_years"] = float(match.group(1)) if match else None
            
        elif line.startswith("Cask Type:"):
            result["cask_type"] = line.replace("Cask Type:", "").strip()
            
        elif line.startswith("Cask Number:"):
            result["cask_number"] = line.replace("Cask Number:", "").strip()
            
        elif "% ABV" in line and "/" in line:
            import re
            abv_match = re.search(r"(\d+\.?\d*)%\s*ABV", line)
            vol_match = re.search(r"/\s*(\d+)cl", line)
            result["abv"] = float(abv_match.group(1)) if abv_match else None
            result["volume_cl"] = int(vol_match.group(1)) if vol_match else None
            
        elif line.startswith("Bottle Number:"):
            raw = line.replace("Bottle Number:", "").strip()
            parts = raw.split("/")
            if len(parts) == 2:
                try:
                    result["bottle_number"] = int(parts[0].strip())
                    result["total_bottles"] = int(parts[1].strip())
                except ValueError:
                    pass

    return result

lot = parse_lot(soup, 867123)
for key, value in lot.items():
    print(f"{key:20s}  {value}")

lot_id                867123
title                 2Bar Spirits 2018 4 Year Old Bourbon Red Wine Barrel American Single Cask #0005 75cl
lot_number            177-05023
winning_bid           40.0
distilled_date        10.31.2018
bottled_date          05.01.2023
age_years             4.0
cask_type             Red Wine Barrel Finish
cask_number           0005
abv                   58.94
volume_cl             75
bottle_number         36
total_bottles         286


In [17]:
soup2 = fetch_lot(867134)
time.sleep(1)

lot2 = parse_lot(soup2, 867134)
for key, value in lot2.items():
    print(f"{key:20s}  {value}")

lot_id                867134
title                 Springbank 2017 8 Year Old Society Bottling
lot_number            177-05034
winning_bid           95.0
distilled_date        February 2017
bottled_date          August 2025
age_years             8.0
cask_type             Fresh Madeira
cask_number           None
abv                   54.7
volume_cl             70
bottle_number         None
total_bottles         None


In [18]:
def get_or_create(cursor, table, name_col, name_value):
    """
    Look up a record by name. Insert it if it doesn't exist.
    Always returns the id.
    """
    cursor.execute(
        f"SELECT id FROM {table} WHERE {name_col} = ?", 
        (name_value,)
    )
    row = cursor.fetchone()
    if row:
        return row[0]
    cursor.execute(
        f"INSERT INTO {table} ({name_col}) VALUES (?)", 
        (name_value,)
    )
    return cursor.lastrowid

def insert_lot(cursor, lot, auction_id):
    """
    Insert a parsed lot dictionary into the database.
    Handles lookup table resolution automatically.
    """
    cask_type_id = None
    if lot["cask_type"]:
        cask_type_id = get_or_create(cursor, "cask_types", "name", lot["cask_type"])

    cursor.execute("""
        INSERT OR IGNORE INTO lots (
            lot_id, lot_number, auction_id,
            cask_type_id, title, winning_bid,
            distilled_date, bottled_date, age_years,
            abv, volume_cl, bottle_number,
            total_bottles, cask_number
        ) VALUES (
            :lot_id, :lot_number, :auction_id,
            :cask_type_id, :title, :winning_bid,
            :distilled_date, :bottled_date, :age_years,
            :abv, :volume_cl, :bottle_number,
            :total_bottles, :cask_number
        )
    """, {
        **lot,
        "auction_id":   auction_id,
        "cask_type_id": cask_type_id,
    })

    return cursor.lastrowid

In [19]:
cursor.execute("SELECT id FROM auctions WHERE auction_number = 177")
auction_id = cursor.fetchone()[0]

insert_lot(cursor, lot2, auction_id)
conn.commit()

cursor.execute("SELECT lot_id, title, winning_bid, cask_type_id FROM lots")
for row in cursor.fetchall():
    print(row)

(867123, '2Bar Spirits 2018 4 Year Old Bourbon Red Wine Barrel American Single Cask #0005 75cl', 40.0, 1)
(867134, 'Springbank 2017 8 Year Old Society Bottling', 95.0, 2)


In [20]:
cursor.execute("SELECT * FROM cask_types")
print(cursor.fetchall())

[(1, 'Red Wine Barrel Finish'), (2, 'Fresh Madeira')]


In [21]:
def scrape_auction(cursor, auction_id, auction_slug, 
                   id_start, id_end, delay=1.5):
    
    scraped = 0
    skipped = 0
    failed  = 0
    
    for lot_id in range(id_start, id_end + 1):
        try:
            cursor.execute(
                "SELECT id FROM lots WHERE lot_id = ?", (lot_id,)
            )
            if cursor.fetchone():
                skipped += 1
                continue
            
            soup = fetch_lot(lot_id, auction_slug)
            time.sleep(delay)
            
            if soup is None:
                failed += 1
                continue
                
            if "Winning bid" not in soup.get_text():
                failed += 1
                continue
            
            lot = parse_lot(soup, lot_id)
            insert_lot(cursor, lot, auction_id)
            conn.commit()
            scraped += 1
            
            if scraped % 10 == 0:
                print(f"  scraped={scraped} skipped={skipped} failed={failed} "
                      f"current={lot_id}")
                
        except Exception as e:
            print(f"  ERROR on lot {lot_id}: {e}")
            failed += 1
            continue
    
    print(f"\nDone. scraped={scraped} skipped={skipped} failed={failed}")

In [22]:
cursor.execute(
    "SELECT id FROM auctions WHERE auction_number = 177"
)
auction_id = cursor.fetchone()[0]

print(f"Testing scraper on 10 lots...")
print(f"Auction id: {auction_id}")
print()

scrape_auction(
    cursor=cursor,
    auction_id=auction_id,
    auction_slug="226-the-177th-auction",
    id_start=867120,
    id_end=867130,
    delay=1.5
)

Testing scraper on 10 lots...
Auction id: 1

  scraped=10 skipped=1 failed=0 current=867130

Done. scraped=10 skipped=1 failed=0


In [23]:
cursor.execute("""
    SELECT l.lot_id, l.title, l.winning_bid, l.age_years, 
           l.abv, l.volume_cl, ct.name as cask_type
    FROM lots l
    LEFT JOIN cask_types ct ON l.cask_type_id = ct.id
    ORDER BY l.lot_id
""")

rows = cursor.fetchall()
print(f"Total lots in database: {len(rows)}\n")
for row in rows:
    lot_id, title, bid, age, abv, vol, cask = row
    print(f"{lot_id}  £{bid or 0:>6.0f}  "
          f"age={str(age or '?'):>4}  "
          f"abv={str(abv or '?'):>5}  "
          f"{(cask or 'unknown'):25s}  "
          f"{title[:45]}")

Total lots in database: 12

867120  £    50  age= 6.0  abv= 65.0  New American Oak           Santa Fe Spirits 2017 6 Year Old Mesquite Smo
867121  £    55  age= 6.0  abv= 65.0  New American Oak           Santa Fe Spirits 2017 6 Year Old Mesquite Smo
867122  £    40  age= 4.0  abv=58.94  Red Wine Barrel Finish     2Bar Spirits 2018 4 Year Old Bourbon Red Wine
867123  £    40  age= 4.0  abv=58.94  Red Wine Barrel Finish     2Bar Spirits 2018 4 Year Old Bourbon Red Wine
867124  £    40  age= 4.0  abv=58.94  Red Wine Barrel Finish     2Bar Spirits 2018 4 Year Old Bourbon Red Wine
867125  £    40  age= 6.0  abv=63.25  New Oak , Heavy Char       Heritage Distilling Co. 2015 6 Year Old 100% 
867126  £    40  age= 6.0  abv=63.25  New Oak , Heavy Char       Heritage Distilling Co. 2015 6 Year Old 100% 
867127  £    40  age= 6.0  abv=63.25  New Oak , Heavy Char       Heritage Distilling Co. 2015 6 Year Old 100% 
867128  £    35  age= 6.0  abv=62.45  2nd Fill Single Malt       Copperworks Distill

In [24]:
response = requests.get("https://www.scotchwhiskyauctions.com/auctions/")
soup = BeautifulSoup(response.text, "html.parser")

auction_links = soup.find_all("a", href=True)
auction_links = [a for a in auction_links 
                 if "/auctions/" in a["href"]
                 and a["href"] != "/auctions/"
                 and "current" not in a["href"]
                 and "#" not in a["href"]]

auctions = []
for link in auction_links:
    href = link["href"]
    text = link.text.strip()
    parts = href.strip("/").split("/")
    if len(parts) >= 2:
        slug = parts[1]
        auctions.append({"slug": slug, "text": text[:50]})

print(f"Auctions found: {len(auctions)}")
for a in auctions[:10]:
    print(f"  {a['slug']:45s}  {a['text'][:40]}")

Auctions found: 179
  226-the-177th-auction                          The 177th AuctionEnds March 8, 2026There
  225-the-176th-auction                          The 176th AuctionEnded February 8, 2026T
  224-the-175th-auction                          The 175th AuctionEnded January 11, 2026T
  223-the-174th-auction                          The 174th AuctionEnded December 7, 2025T
  222-the-173rd-auction                          The 173rd AuctionEnded November 9, 2025T
  221-the-172nd-auction                          The 172nd AuctionEnded October 12, 2025T
  220-the-171st-auction                          The 171st AuctionEnded September 14, 202
  219-the-170th-auction                          The 170th AuctionEnded August 10, 2025Th
  218-the-169th-auction                          The 169th AuctionEnded July 13, 2025Ther
  217-the-168th-auction                          The 168th AuctionEnded June 8, 2025There


In [25]:
import re

def get_auction_number(slug):
    match = re.search(r"the-(\d+)", slug)
    return int(match.group(1)) if match else None

def probe_auction_range(slug, sample_size=3, delay=1.5):
    """
    Fetch the first page of an auction listing and extract
    a few lot IDs to estimate the ID range.
    """
    url = f"https://www.scotchwhiskyauctions.com/auctions/{slug}/"
    r = requests.get(url, headers={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    })
    time.sleep(delay)
    
    if r.status_code != 200:
        return None, None
    
    soup = BeautifulSoup(r.text, "html.parser")
    lot_links = [
        a["href"] for a in soup.find_all("a", href=True)
        if f"/auctions/{slug}/" in a["href"]
        and "#" not in a["href"]
        and a["href"] != f"/auctions/{slug}/"
    ]
    
    ids = []
    for link in lot_links:
        parts = link.strip("/").split("/")
        if len(parts) >= 3:
            try:
                lot_id = int(parts[2].split("-")[0])
                ids.append(lot_id)
            except ValueError:
                continue
    
    if ids:
        return min(ids), max(ids)
    return None, None

min_id, max_id = probe_auction_range("226-the-177th-auction")
print(f"Auction 177: lot IDs {min_id} to {max_id}")

Auction 177: lot IDs 862100 to 867182


In [26]:
test_slugs = [
    "225-the-176th-auction",
    "224-the-175th-auction",
    "220-the-171st-auction",
    "210-the-162nd-auction",
]

for slug in test_slugs:
    min_id, max_id = probe_auction_range(slug)
    num = get_auction_number(slug)
    print(f"Auction {num:3d}: {min_id} to {max_id}")
    time.sleep(1)

Auction 176: 857584 to 861947
Auction 175: 847191 to 857052
Auction 171: 825559 to 835241
Auction 162: 783059 to 789395


In [27]:
def build_auction_registry(auction_list, delay=2.0):
    """
    Probe each auction to find its lot ID range.
    Returns a list of dicts ready to insert into the auctions table.
    """
    registry = []
    
    for i, auction in enumerate(auction_list):
        slug = auction["slug"]
        num = get_auction_number(slug)
        
        if num is None:
            continue
            
        min_id, max_id = probe_auction_range(slug, delay=delay)
        
        if min_id is None:
            print(f"  [{i+1:3d}/{len(auction_list)}] Auction {num:3d}: no lots found")
            continue
        
        registry.append({
            "auction_number": num,
            "slug":           slug,
            "id_start":       min_id,
            "id_end":         max_id,
            "url": f"https://www.scotchwhiskyauctions.com/auctions/{slug}/"
        })
        
        print(f"  [{i+1:3d}/{len(auction_list)}] "
              f"Auction {num:3d}: {min_id} to {max_id} "
              f"(range {max_id - min_id:,})")
        
        time.sleep(delay)
    
    return registry

recent_auctions = [a for a in auctions 
                   if get_auction_number(a["slug"]) is not None
                   and get_auction_number(a["slug"]) >= 154]

print(f"Auctions to probe: {len(recent_auctions)}")
print("Starting registry build (this will take ~1 minute)...\n")

registry = build_auction_registry(recent_auctions, delay=2.0)
print(f"\nRegistry complete: {len(registry)} auctions mapped")

Auctions to probe: 24
Starting registry build (this will take ~1 minute)...

  [  1/24] Auction 177: 862100 to 867182 (range 5,082)
  [  2/24] Auction 176: 857584 to 861947 (range 4,363)
  [  3/24] Auction 175: 847191 to 857052 (range 9,861)
  [  4/24] Auction 174: 847301 to 853007 (range 5,706)
  [  5/24] Auction 173: 841368 to 846899 (range 5,531)
  [  6/24] Auction 172: 830552 to 840988 (range 10,436)
  [  7/24] Auction 171: 825559 to 835241 (range 9,682)
  [  8/24] Auction 170: 825472 to 829675 (range 4,203)
  [  9/24] Auction 169: 815056 to 823171 (range 8,115)
  [ 10/24] Auction 168: 814886 to 819058 (range 4,172)
  [ 11/24] Auction 167: 809685 to 814858 (range 5,173)
  [ 12/24] Auction 166: 803706 to 809705 (range 5,999)
  [ 13/24] Auction 165: 798540 to 803511 (range 4,971)
  [ 14/24] Auction 164: 790441 to 798341 (range 7,900)
  [ 15/24] Auction 163: 790354 to 794375 (range 4,021)
  [ 16/24] Auction 162: 783059 to 789395 (range 6,336)
  [ 17/24] Auction 161: 775956 to 782840 (

In [28]:
total_ids = sum(r["id_end"] - r["id_start"] for r in registry)
estimated_lots = int(total_ids * 0.93)
scrape_time_hours = (total_ids * 1.5) / 3600

print(f"Total ID range to iterate: {total_ids:,}")
print(f"Estimated valid lots (~93%): {estimated_lots:,}")
print(f"Estimated scrape time at 1.5s: {scrape_time_hours:.1f} hours")
print(f"\nBreakdown by auction:")
for r in registry[:5]:
    print(f"  Auction {r['auction_number']:3d}: "
          f"{r['id_start']:,} to {r['id_end']:,} "
          f"= {r['id_end']-r['id_start']:,} IDs")
print("  ...")

Total ID range to iterate: 148,399
Estimated valid lots (~93%): 138,011
Estimated scrape time at 1.5s: 61.8 hours

Breakdown by auction:
  Auction 177: 862,100 to 867,182 = 5,082 IDs
  Auction 176: 857,584 to 861,947 = 4,363 IDs
  Auction 175: 847,191 to 857,052 = 9,861 IDs
  Auction 174: 847,301 to 853,007 = 5,706 IDs
  Auction 173: 841,368 to 846,899 = 5,531 IDs
  ...


In [29]:
for r in registry:
    cursor.execute("""
        INSERT OR IGNORE INTO auctions (auction_number, url)
        VALUES (:auction_number, :url)
    """, r)

conn.commit()

cursor.execute("SELECT COUNT(*) FROM auctions")
print(f"\nAuctions in database: {cursor.fetchone()[0]}")


Auctions in database: 24


In [30]:
total_ids_optimised = sum(r["id_end"] - r["id_start"] for r in registry)
scrape_time_optimised = (total_ids_optimised * 1.2) / 3600
print(f"At 1.2s delay: {scrape_time_optimised:.1f} hours")

At 1.2s delay: 49.5 hours


In [34]:
scraper_code = '''# -*- coding: utf-8 -*-
import requests
import sqlite3
import time
import re
from bs4 import BeautifulSoup
from datetime import datetime

DB_PATH = "whisky_auctions.db"
DELAY   = 1.2

def fetch_lot(lot_id, auction_slug):
    url = f"https://www.scotchwhiskyauctions.com/auctions/{auction_slug}/{lot_id}/"
    try:
        r = requests.get(url, headers={
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        }, timeout=10)
        if r.status_code == 200:
            return BeautifulSoup(r.text, "html.parser")
    except Exception:
        pass
    return None

def parse_lot(soup, lot_id):
    result = {
        "lot_id": lot_id, "title": None, "lot_number": None,
        "winning_bid": None, "distilled_date": None, "bottled_date": None,
        "age_years": None, "cask_type": None, "cask_number": None,
        "abv": None, "volume_cl": None, "bottle_number": None,
        "total_bottles": None,
    }
    
    h1 = soup.find("h1")
    result["title"] = h1.text.strip() if h1 else None
    
    lines = [l.strip() for l in 
             soup.get_text(separator="\\n").split("\\n") if l.strip()]
    
    for line in lines:
        if line.startswith("Lot number:"):
            result["lot_number"] = line.replace("Lot number:", "").strip()
        elif line.startswith("Winning bid:"):
            raw = line.replace("Winning bid:", "").replace("£","").replace(",","").strip()
            try:
                result["winning_bid"] = float(raw)
            except ValueError:
                pass
        elif line.startswith("Distilled:"):
            result["distilled_date"] = line.replace("Distilled:", "").strip()
        elif line.startswith("Bottled:"):
            result["bottled_date"] = line.replace("Bottled:", "").strip()
        elif line.startswith("Age:"):
            m = re.search(r"(\\d+\\.?\\d*)", line)
            result["age_years"] = float(m.group(1)) if m else None
        elif line.startswith("Cask Type:"):
            result["cask_type"] = line.replace("Cask Type:", "").strip()
        elif line.startswith("Cask Number:"):
            result["cask_number"] = line.replace("Cask Number:", "").strip()
        elif "% ABV" in line and "/" in line:
            abv = re.search(r"(\\d+\\.?\\d*)%\\s*ABV", line)
            vol = re.search(r"/\\s*(\\d+)cl", line)
            result["abv"]       = float(abv.group(1)) if abv else None
            result["volume_cl"] = int(vol.group(1))   if vol else None
        elif line.startswith("Bottle Number:"):
            parts = line.replace("Bottle Number:", "").strip().split("/")
            if len(parts) == 2:
                try:
                    result["bottle_number"] = int(parts[0].strip())
                    result["total_bottles"] = int(parts[1].strip())
                except ValueError:
                    pass
    return result

def get_or_create(cursor, table, col, val):
    cursor.execute(f"SELECT id FROM {table} WHERE {col} = ?", (val,))
    row = cursor.fetchone()
    if row:
        return row[0]
    cursor.execute(f"INSERT INTO {table} ({col}) VALUES (?)", (val,))
    return cursor.lastrowid

def insert_lot(cursor, lot, auction_id):
    cask_type_id = None
    if lot["cask_type"]:
        cask_type_id = get_or_create(cursor, "cask_types", "name", lot["cask_type"])
    cursor.execute("""
        INSERT OR IGNORE INTO lots (
            lot_id, lot_number, auction_id, cask_type_id,
            title, winning_bid, distilled_date, bottled_date,
            age_years, abv, volume_cl, bottle_number,
            total_bottles, cask_number
        ) VALUES (
            :lot_id, :lot_number, :auction_id, :cask_type_id,
            :title, :winning_bid, :distilled_date, :bottled_date,
            :age_years, :abv, :volume_cl, :bottle_number,
            :total_bottles, :cask_number
        )
    """, {**lot, "auction_id": auction_id, "cask_type_id": cask_type_id})

def run():
    conn   = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    cursor.execute("""
        SELECT id, auction_number, url FROM auctions
        ORDER BY auction_number DESC
    """)
    auction_rows = cursor.fetchall()
    
    auction_registry = {
        177: (862100, 867182, "226-the-177th-auction"),
        176: (857584, 861947, "225-the-176th-auction"),
        175: (847191, 857052, "224-the-175th-auction"),
        174: (847301, 853007, "223-the-174th-auction"),
        173: (841368, 846899, "222-the-173rd-auction"),
        172: (830552, 840988, "221-the-172nd-auction"),
        171: (825559, 835241, "220-the-171st-auction"),
        170: (825472, 829675, "219-the-170th-auction"),
        169: (815056, 823171, "218-the-169th-auction"),
        168: (814886, 819058, "217-the-168th-auction"),
        167: (809685, 814858, "216-the-167th-auction"),
        166: (803706, 809705, "215-the-166th-auction"),
        165: (798540, 803511, "214-the-165th-auction"),
        164: (790441, 798341, "213-the-164th-auction"),
        163: (790354, 794375, "212-the-163rd-auction"),
        162: (783059, 789395, "210-the-162nd-auction"),
        161: (775956, 782840, "209-the-161st-auction"),
        160: (766795, 775801, "208-the-160th-auction"),
        159: (762179, 768111, "207-the-159th-auction"),
        158: (756745, 761774, "206-the-158th-auction"),
        157: (751345, 754921, "205-the-157th-auction"),
        156: (746236, 750997, "204-the-156th-auction"),
        155: (740882, 745356, "203-the-155th-auction"),
        154: (732706, 739892, "202-the-154th-auction"),
    }
    
    total_scraped = 0
    total_skipped = 0
    total_failed  = 0
    
    for auction_db_id, auction_number, url in auction_rows:
        if auction_number not in auction_registry:
            continue
            
        id_start, id_end, slug = auction_registry[auction_number]
        
        print(f"\\n[{datetime.now():%H:%M:%S}] "
              f"Auction {auction_number} ({slug})")
        print(f"  ID range: {id_start:,} to {id_end:,} "
              f"({id_end - id_start:,} IDs)")
        
        scraped = skipped = failed = 0
        
        for lot_id in range(id_start, id_end + 1):
            try:
                cursor.execute(
                    "SELECT id FROM lots WHERE lot_id = ?", (lot_id,)
                )
                if cursor.fetchone():
                    skipped += 1
                    continue
                
                soup = fetch_lot(lot_id, slug)
                time.sleep(DELAY)
                
                if not soup or "Winning bid" not in soup.get_text():
                    failed += 1
                    continue
                
                lot = parse_lot(soup, lot_id)
                insert_lot(cursor, lot, auction_db_id)
                conn.commit()
                scraped += 1
                
                if scraped % 100 == 0:
                    print(f"  [{datetime.now():%H:%M:%S}] "
                          f"scraped={scraped:,} "
                          f"skipped={skipped:,} "
                          f"failed={failed:,} "
                          f"lot={lot_id:,}")
                          
            except KeyboardInterrupt:
                print("\\nInterrupted — progress saved.")
                conn.close()
                return
            except Exception as e:
                failed += 1
                continue
        
        total_scraped += scraped
        total_skipped += skipped
        total_failed  += failed
        
        print(f"  Done: scraped={scraped:,} "
              f"skipped={skipped:,} failed={failed:,}")
    
    print(f"\\nAll auctions complete.")
    print(f"Total: scraped={total_scraped:,} "
          f"skipped={total_skipped:,} failed={total_failed:,}")
    conn.close()

if __name__ == "__main__":
    run()
'''

with open("scraper.py", "w") as f:
    f.write(scraper_code)

print("scraper.py written successfully")

scraper.py written successfully


In [35]:
cursor.execute("""
    SELECT 
        COUNT(*)                                    as total,
        SUM(CASE WHEN winning_bid IS NULL THEN 1 ELSE 0 END)  as no_price,
        SUM(CASE WHEN age_years   IS NULL THEN 1 ELSE 0 END)  as no_age,
        SUM(CASE WHEN abv         IS NULL THEN 1 ELSE 0 END)  as no_abv,
        SUM(CASE WHEN cask_type_id IS NULL THEN 1 ELSE 0 END) as no_cask,
        ROUND(AVG(winning_bid), 2)                  as avg_price,
        MIN(winning_bid)                            as min_price,
        MAX(winning_bid)                            as max_price
    FROM lots
    WHERE lot_id > 862100
""")

row = cursor.fetchone()
cols = ["total", "no_price", "no_age", "no_abv", 
        "no_cask", "avg_price", "min_price", "max_price"]

for col, val in zip(cols, row):
    print(f"{col:12s}  {val}")

total         272
no_price      0
no_age        102
no_abv        1
no_cask       96
avg_price     171.32
min_price     15.0
max_price     4000.0


In [36]:
cursor.execute("""
    SELECT l.lot_id, l.title, l.winning_bid, l.age_years, 
           l.abv, ct.name as cask_type
    FROM lots l
    LEFT JOIN cask_types ct ON l.cask_type_id = ct.id
    WHERE l.lot_id > 862100
    ORDER BY l.winning_bid DESC
    LIMIT 15
""")

rows = cursor.fetchall()
for row in rows:
    lot_id, title, bid, age, abv, cask = row
    print(f"£{bid or 0:>7,.0f}  "
          f"age={str(age or '?'):>4}  "
          f"{(cask or 'unknown'):20s}  "
          f"{title[:45]}")

£  4,000  age=25.0  Sherry Butt           Ardbeg 1976 Single Cask #2390 Hand Filled
£  2,600  age=18.0  Sherry Hogshead       Arran White Stag First Release
£  1,800  age=   ?  unknown               Mortlach 1954 Gordon & MacPhail Private Colle
£  1,800  age=25.0  unknown               Bowmore 25 Year Old Chateau Lagrange Auld All
£  1,200  age=   ?  Sherry Butt           Karuizawa 2000 Sea Dragon Cask #166
£  1,100  age=33.0  Bourbon / White Tawny Port  Bowmore 33 Year Old The Changeling
£    850  age=25.0  unknown               Springbank 25 Year Old 2021 Release
£    850  age=   ?  unknown               Macallan Edition No1
£    850  age=   ?  unknown               Macallan Edition No1
£    800  age=32.0  Oloroso Sherry Finish  Tobermory 1972 32 Year Old
£    750  age=30.0  unknown               Bowmore 30 Year Old 2024 Release
£    750  age=   ?  unknown               Highland Park 1970 Rare Old Gordon & MacPhail
£    650  age=30.0  Sherry                Glengoyne 30 Year Old 2018 

In [37]:
conn.close()
print("Database connection closed")

Database connection closed


In [38]:
KNOWN_BOTTLERS = [
    "Gordon & MacPhail", "G&M",
    "Cadenhead's", "Cadenheads", "Cadenhead",
    "Signatory", "Signatory Vintage",
    "Scott's Selection", "Scotts Selection",
    "Berry Bros", "Berry Brothers", "Berry Bros & Rudd",
    "Duncan Taylor",
    "Douglas Laing", "Douglas Laing & Co",
    "Old Malt Cask", "Old & Rare",
    "Platinum Old & Rare",
    "Hunter Laing",
    "First Editions",
    "SMWS", "Scotch Malt Whisky Society",
    "Adelphi",
    "Hart Brothers",
    "Murray McDavid",
    "Blackadder",
    "Compass Box",
    "Kingsbury",
    "Whisky Agency",
    "Malts of Scotland",
    "Liquid Treasures",
    "Sansibar",
    "Chieftain's",
    "Ian Macleod",
    "James MacArthur",
    "Dun Bheagan",
    "Wilson & Morgan",
    "Samaroli",
    "Silver Seal",
    "Moon Import",
    "Sestante",
    "American Single Cask",
]

def extract_bottler(title):
    """
    Check if any known bottler name appears in the lot title.
    Returns the bottler name if found, None otherwise.
    """
    if not title:
        return None
    title_lower = title.lower()
    for bottler in KNOWN_BOTTLERS:
        if bottler.lower() in title_lower:
            return bottler
    return None

def is_ob(title, bottler):
    """
    A lot is an official bottling if no known IB is detected.
    This is a heuristic — not perfect but good enough for initial analysis.
    """
    if bottler:
        return False
    return True

test_titles = [
    "Springbank 25 Year Old 2021 Release",
    "Mortlach 1954 Gordon & MacPhail Private Collection",
    "Ardbeg 1976 Single Cask #2390 Hand Filled",
    "Caol Ila 1979 Signatory Vintage 23 Year Old",
    "GlenDronach 1993 Single Cask #180 28 Year Old",
    "Karuizawa 2000 Sea Dragon Cask #166",
    "Bowmore 1966 Berry Bros & Rudd 38 Year Old",
]

print(f"{'Title':50s}  {'Bottler':25s}  {'OB?'}")
print("-" * 85)
for title in test_titles:
    bottler = extract_bottler(title)
    ob = is_ob(title, bottler)
    print(f"{title[:50]:50s}  {(bottler or 'none'):25s}  {ob}")

Title                                               Bottler                    OB?
-------------------------------------------------------------------------------------
Springbank 25 Year Old 2021 Release                 none                       True
Mortlach 1954 Gordon & MacPhail Private Collection  Gordon & MacPhail          False
Ardbeg 1976 Single Cask #2390 Hand Filled           none                       True
Caol Ila 1979 Signatory Vintage 23 Year Old         Signatory                  False
GlenDronach 1993 Single Cask #180 28 Year Old       none                       True
Karuizawa 2000 Sea Dragon Cask #166                 none                       True
Bowmore 1966 Berry Bros & Rudd 38 Year Old          Berry Bros                 False


In [39]:
type scrape_log.txt

SyntaxError: invalid syntax (3277451282.py, line 1)

The closed distillery premium. Karuizawa, Port Ellen, Brora, Rosebank, Littlemill, St Magdalene — bottles from closed distilleries should show a dramatically different price curve to operational ones. And crucially, the premium should be accelerating over time as remaining stock dwindles. You might be able to quantify the rate at which that premium grows per year.

The Italian bottler effect. Samaroli and Moon Import bottlings from the 1970s and 80s are legendary but that reputation is partly connoisseur mythology and partly real quality. Your regression model will tell you whether a Samaroli label actually commands a statistically significant premium controlling for distillery, age, and vintage — or whether it's priced on reputation alone.

The GlenDronach 19/3/1993 question. Once you have enough GlenDronach lots you can literally test whether that date is statistically significant. My prediction is it will be — but the coefficient will be smaller than the whisky community believes because some of the premium is confirmation bias from enthusiasts who already know the bottle's reputation before tasting it.

The Macallan quality cliff. Your personal tasting experience suggested a quality break around 2015-2017. If that's real and not just your palate, you should see it as a discontinuity in the price-per-year-of-age curve for Macallan lots grouped by distillation year. A structural break test on that series would be a genuinely publishable finding in whisky economics.

The ABV premium. Cask strength bottlings should command a premium over diluted versions of the same whisky. But how large is that premium per percentage point of ABV? And does it interact with age — are collectors more willing to pay for cask strength on older expressions?


In [40]:
conn2 = sqlite3.connect("whisky_auctions.db")
c2 = conn2.cursor()

c2.execute("""
    SELECT 
        COUNT(*)                                          as total,
        SUM(CASE WHEN winning_bid IS NULL THEN 1 ELSE 0 END)   as no_price,
        SUM(CASE WHEN age_years IS NULL THEN 1 ELSE 0 END)     as no_age,
        SUM(CASE WHEN abv IS NULL THEN 1 ELSE 0 END)           as no_abv,
        SUM(CASE WHEN distilled_date IS NULL THEN 1 ELSE 0 END) as no_distilled,
        ROUND(AVG(winning_bid), 2)                        as avg_price,
        MAX(winning_bid)                                  as max_price,
        MIN(winning_bid)                                  as min_price
    FROM lots
""")

row = c2.fetchone()
cols = ["total", "no_price", "no_age", "no_abv", 
        "no_distilled", "avg_price", "max_price", "min_price"]

print("Data quality check:")
print("-" * 35)
for col, val in zip(cols, row):
    pct = f"  ({100*val/row[0]:.1f}%)" if col.startswith("no_") else ""
    print(f"  {col:15s}  {val}{pct}")

Data quality check:
-----------------------------------
  total            7244
  no_price         0  (0.0%)
  no_age           2956  (40.8%)
  no_abv           69  (1.0%)
  no_distilled     4960  (68.5%)
  avg_price        173.76
  max_price        20000.0
  min_price        15.0


In [41]:
c2.execute("""
    SELECT l.lot_id, l.title, l.winning_bid, 
           l.age_years, l.distilled_date,
           ct.name as cask_type
    FROM lots l
    LEFT JOIN cask_types ct ON l.cask_type_id = ct.id
    WHERE l.winning_bid IS NOT NULL
    ORDER BY l.winning_bid DESC
    LIMIT 20
""")

rows = c2.fetchall()
print(f"\nTop 20 most expensive lots so far:\n")
print(f"{'Price':>8}  {'Age':>4}  {'Distilled':12}  {'Title'}")
print("-" * 75)
for lot_id, title, bid, age, dist, cask in rows:
    print(f"£{bid:>7,.0f}  "
          f"{str(age or '?'):>4}  "
          f"{(dist or 'unknown'):12}  "
          f"{title[:45]}")

conn2.close()


Top 20 most expensive lots so far:

   Price   Age  Distilled     Title
---------------------------------------------------------------------------
£ 20,000  48.0  01.09.1964    Karuizawa 1964 48 Year Old Wealth Solutions
£ 13,000  53.0  1949          Macallan Fine & Rare 1949 53 Year Old
£ 11,000  50.0  1937          Balvenie 1937 50 Year Old 75cl
£ 10,000  40.0  unknown       Macallan 40 Year Old The Red Collection
£  8,500  50.0  December 1926  Dalmore 1926 50 Year Old
£  4,800     ?  unknown       Macallan Cask 888
£  4,600     ?  unknown       Macallan Distil Your World Paris Edition
£  4,000  25.0  24.11.1976    Ardbeg 1976 Single Cask #2390 Hand Filled
£  3,800  15.0  1952          Macallan 1952 Over 15 Year Old Hope & King 80
£  3,800  33.0  1974 & 1975   Ardbeg 1815
£  3,600  12.0  1995 / 1996   Macallan Easter Elchies 2008
£  3,400     ?  unknown       Macallan Distil Your World London Edition
£  3,200     ?  unknown       Macallan Distil Your World London Edition
£  3,200  

In [42]:
conn2 = sqlite3.connect("whisky_auctions.db")
c2 = conn2.cursor()

c2.execute("""
    SELECT 
        a.auction_number,
        COUNT(l.id)              as lots,
        ROUND(AVG(l.winning_bid), 2)  as avg_price,
        MAX(l.winning_bid)       as max_price,
        ROUND(AVG(l.age_years), 1)    as avg_age
    FROM lots l
    JOIN auctions a ON l.auction_id = a.id
    GROUP BY a.auction_number
    ORDER BY a.auction_number DESC
""")

rows = c2.fetchall()
print(f"{'Auction':>8}  {'Lots':>6}  {'Avg £':>8}  {'Max £':>8}  {'Avg age':>8}")
print("-" * 50)
for auction, lots, avg, max_p, age in rows:
    print(f"{auction:>8}  {lots:>6,}  "
          f"£{avg or 0:>7,.0f}  "
          f"£{max_p or 0:>7,.0f}  "
          f"{str(age or '?'):>8}")

conn2.close()

 Auction    Lots     Avg £     Max £   Avg age
--------------------------------------------------
     177   4,769  £    183  £ 20,000      16.0
     176   4,170  £    169  £  8,500      15.8
     175   3,906  £    185  £ 10,000      15.8
     174   5,442  £    182  £ 36,000      16.1
     173   5,226  £    172  £  9,500      16.1
     172   5,370  £    181  £ 14,500      16.0
     171   1,952  £    176  £ 25,000      15.7


In [43]:
conn2 = sqlite3.connect("whisky_auctions.db")
c2 = conn2.cursor()

c2.execute("""
    SELECT l.lot_id, l.title, l.winning_bid, 
           l.age_years, l.distilled_date, l.abv
    FROM lots l
    JOIN auctions a ON l.auction_id = a.id
    WHERE a.auction_number = 174
    ORDER BY l.winning_bid DESC
    LIMIT 10
""")

rows = c2.fetchall()
print("Top 10 most expensive lots in auction 174:\n")
for lot_id, title, bid, age, dist, abv in rows:
    print(f"£{bid:>8,.0f}  age={str(age or '?'):>4}  "
          f"{(dist or 'unknown'):15}  {title[:50]}")

conn2.close()

Top 10 most expensive lots in auction 174:

£  36,000  age=52.0  unknown          Macallan 52 Year Old 2018 Release
£  11,500  age=40.0  unknown          Macallan 40 Year Old The Red Collection
£  11,000  age=42.0  05.11.1964       Bowmore Black 1964 42 Year Old 75cl
£  11,000  age=42.0  05.11.1964       Bowmore Black 1964 42 Year Old 75cl
£   7,500  age=   ?  unknown          Macallan The Archival Series - Folio 1
£   5,400  age=   ?  unknown          Hakushu ‘First Class’ for Moegi no Mura
£   4,200  age=40.0  unknown          Bowmore 40 Year Old 2022 Release
£   3,800  age=   ?  1959             Glenfiddich 1959 Private Vintage Single Cask #3934
£   3,800  age=   ?  1958             Glenfiddich 1958 Private Vintage Single Cask #8642
£   3,800  age=   ?  27.11.2008       Macallan 2008 Distil Your World London Single Cask


In [44]:
conn2 = sqlite3.connect("whisky_auctions.db")
c2 = conn2.cursor()

c2.execute("""
    SELECT 
        COUNT(*)                                            as total,
        SUM(CASE WHEN winning_bid IS NULL  THEN 1 ELSE 0 END)  as no_price,
        SUM(CASE WHEN age_years IS NULL    THEN 1 ELSE 0 END)  as no_age,
        SUM(CASE WHEN distilled_date IS NULL THEN 1 ELSE 0 END) as no_dist,
        SUM(CASE WHEN abv IS NULL          THEN 1 ELSE 0 END)  as no_abv,
        SUM(CASE WHEN cask_type_id IS NULL THEN 1 ELSE 0 END)  as no_cask,
        ROUND(AVG(winning_bid), 2)                          as avg_price,
        ROUND(MIN(winning_bid), 2)                          as min_price,
        ROUND(MAX(winning_bid), 2)                          as max_price,
        ROUND(AVG(age_years), 1)                            as avg_age
    FROM lots
""")

row = c2.fetchone()
cols = ["total", "no_price", "no_age", "no_dist", 
        "no_abv", "no_cask", "avg_price", 
        "min_price", "max_price", "avg_age"]

print("Full dataset quality check:")
print("-" * 40)
for col, val in zip(cols, row):
    if col.startswith("no_"):
        pct = 100 * val / row[0]
        print(f"  {col:12s}  {val:>6,}  ({pct:.1f}%)")
    else:
        print(f"  {col:12s}  {val}")

Full dataset quality check:
----------------------------------------
  total         30985
  no_price           0  (0.0%)
  no_age        12,963  (41.8%)
  no_dist       21,356  (68.9%)
  no_abv           328  (1.1%)
  no_cask       15,249  (49.2%)
  avg_price     178.6
  min_price     15.0
  max_price     36000.0
  avg_age       16.0


In [45]:
conn2 = sqlite3.connect("whisky_auctions.db")
c2 = conn2.cursor()

c2.execute("""
    SELECT 
        SUM(CASE WHEN winning_bid < 50     THEN 1 ELSE 0 END) as under_50,
        SUM(CASE WHEN winning_bid BETWEEN 50  AND 99   THEN 1 ELSE 0 END) as p50_99,
        SUM(CASE WHEN winning_bid BETWEEN 100 AND 249  THEN 1 ELSE 0 END) as p100_249,
        SUM(CASE WHEN winning_bid BETWEEN 250 AND 499  THEN 1 ELSE 0 END) as p250_499,
        SUM(CASE WHEN winning_bid BETWEEN 500 AND 999  THEN 1 ELSE 0 END) as p500_999,
        SUM(CASE WHEN winning_bid BETWEEN 1000 AND 4999 THEN 1 ELSE 0 END) as p1k_5k,
        SUM(CASE WHEN winning_bid >= 5000  THEN 1 ELSE 0 END) as over_5k,
        COUNT(*) as total
    FROM lots WHERE winning_bid IS NOT NULL
""")

row = c2.fetchone()
labels = ["<£50","£50-99","£100-249","£250-499",
          "£500-999","£1k-5k",">£5k","total"]

print("Price distribution:")
print("-" * 45)
total = row[-1]
for label, val in zip(labels[:-1], row[:-1]):
    pct = 100 * val / total
    bar = "█" * int(pct / 2)
    print(f"  {label:10s}  {val:>6,}  ({pct:4.1f}%)  {bar}")

conn2.close()

Price distribution:
---------------------------------------------
  <£50         6,978  (22.3%)  ███████████
  £50-99      10,241  (32.7%)  ████████████████
  £100-249     9,524  (30.4%)  ███████████████
  £250-499     2,802  ( 8.9%)  ████
  £500-999     1,081  ( 3.5%)  █
  £1k-5k         680  ( 2.2%)  █
  >£5k            26  ( 0.1%)  


In [46]:
conn2 = sqlite3.connect("whisky_auctions.db")
c2 = conn2.cursor()

c2.execute("""
    SELECT l.lot_id, l.title, l.winning_bid,
           l.age_years, l.distilled_date,
           a.auction_number
    FROM lots l
    JOIN auctions a ON l.auction_id = a.id
    WHERE l.winning_bid >= 5000
    ORDER BY l.winning_bid DESC
""")

rows = c2.fetchall()
print(f"Lots over £5,000: {len(rows)}\n")
print(f"{'Price':>8}  {'Age':>4}  {'Auction':>7}  Title")
print("-" * 75)
for lot_id, title, bid, age, dist, auction in rows:
    print(f"£{bid:>7,.0f}  "
          f"{str(age or '?'):>4}  "
          f"     {auction:>3}  "
          f"{title[:50]}")

conn2.close()

Lots over £5,000: 26

   Price   Age  Auction  Title
---------------------------------------------------------------------------
£ 36,000  52.0       174  Macallan 52 Year Old 2018 Release
£ 25,000  50.0       171  Bowmore 1966 50 Year Old
£ 20,000  48.0       177  Karuizawa 1964 48 Year Old Wealth Solutions
£ 14,500     ?       172  Macallan Distil Your World New York, Mexico & Lond
£ 13,000  53.0       177  Macallan Fine & Rare 1949 53 Year Old
£ 11,500  40.0       174  Macallan 40 Year Old The Red Collection
£ 11,000  50.0       177  Balvenie 1937 50 Year Old 75cl
£ 11,000  42.0       174  Bowmore Black 1964 42 Year Old 75cl
£ 11,000  42.0       174  Bowmore Black 1964 42 Year Old 75cl
£ 10,000  40.0       177  Macallan 40 Year Old The Red Collection
£ 10,000  40.0       175  Macallan 40 Year Old The Red Collection
£  9,500  44.0       173  Bowmore Gold 1964 44 Year Old
£  9,000  43.0       173  Bowmore White 1964 43 Year Old
£  8,500  50.0       176  Dalmore 1926 50 Year Old
£  8,0

In [47]:
python -c "
import sqlite3
conn = sqlite3.connect('whisky_auctions.db')
c = conn.cursor()
c.execute('''
    SELECT a.auction_number, COUNT(*) as lots,
           ROUND(AVG(l.winning_bid),2) as avg_price,
           MAX(l.winning_bid) as max_price
    FROM lots l JOIN auctions a ON l.auction_id = a.id
    GROUP BY a.auction_number
    ORDER BY a.auction_number DESC
''')
print(f'{'Auction':>8}  {'Lots':>7}  {'Avg £':>8}  {'Max £':>8}')
print('-' * 40)
for row in c.fetchall():
    print(f'{row[0]:>8}  {row[1]:>7,}  £{row[2] or 0:>7,.0f}  £{row[3] or 0:>7,.0f}')
conn.close()
"

SyntaxError: unterminated string literal (detected at line 1) (1680373696.py, line 1)

In [48]:
import sqlite3

conn2 = sqlite3.connect("whisky_auctions.db")
c2 = conn2.cursor()

c2.execute("""
    SELECT a.auction_number, COUNT(*) as lots,
           ROUND(AVG(l.winning_bid), 2) as avg_price,
           MAX(l.winning_bid) as max_price
    FROM lots l JOIN auctions a ON l.auction_id = a.id
    GROUP BY a.auction_number
    ORDER BY a.auction_number DESC
""")

print(f"{'Auction':>8}  {'Lots':>7}  {'Avg £':>8}  {'Max £':>8}")
print("-" * 42)
for row in c2.fetchall():
    print(f"{row[0]:>8}  {row[1]:>7,}  "
          f"£{row[2] or 0:>7,.0f}  "
          f"£{row[3] or 0:>7,.0f}")

conn2.close()

 Auction     Lots     Avg £     Max £
------------------------------------------
     177    4,769  £    183  £ 20,000
     176    4,170  £    169  £  8,500
     175    3,906  £    185  £ 10,000
     174    5,442  £    182  £ 36,000
     173    5,226  £    172  £  9,500
     172    5,370  £    181  £ 14,500
     171    4,662  £    189  £ 25,000
     170    3,978  £    170  £ 13,000
     169    3,993  £    169  £  6,000
     168    3,919  £    184  £ 12,000
     167    4,759  £    186  £ 11,500
     166    5,579  £    186  £  5,800
     165    4,702  £    193  £ 48,000
     164    3,784  £    210  £  9,000
     163    3,502  £    195  £ 14,500
     162    5,836  £    212  £ 25,000
     161    2,532  £    181  £  7,500


In [49]:
conn2 = sqlite3.connect("whisky_auctions.db")
c2 = conn2.cursor()

c2.execute("""
    SELECT l.lot_id, l.title, l.winning_bid,
           l.age_years, l.distilled_date, l.abv,
           a.auction_number
    FROM lots l
    JOIN auctions a ON l.auction_id = a.id
    WHERE a.auction_number = 165
    ORDER BY l.winning_bid DESC
    LIMIT 5
""")

for row in c2.fetchall():
    lot_id, title, bid, age, dist, abv, auction = row
    print(f"£{bid:>8,.0f}  age={str(age or '?'):>4}  "
          f"dist={str(dist or 'unknown'):15}  {title[:50]}")

conn2.close()

£  48,000  age=72.0  dist=unknown          Macallan 72 Year Old Lalique Genesis Decanter
£   7,000  age=30.0  dist=1990             Glenfarclas 1990 30 Year Old Worlds Series #1 6 x 
£   7,000  age=60.0  dist=03.01.1956       Linkwood 1956 60 Year Old Cask #20 Private Collect
£   6,500  age=   ?  dist=24.12.1953       Glen Grant 1953 Mr George Legacy First Edition Gor
£   6,500  age=   ?  dist=24.12.1953       Glen Grant 1953 Mr George Legacy First Edition Gor


In [50]:
conn2 = sqlite3.connect("whisky_auctions.db")
c2 = conn2.cursor()

c2.execute("""
    SELECT 
        a.auction_number,
        COUNT(*)              as lots_scraped,
        MIN(l.lot_id)         as min_lot_id,
        MAX(l.lot_id)         as max_lot_id,
        MAX(l.scraped_at)     as last_scraped
    FROM lots l
    JOIN auctions a ON l.auction_id = a.id
    WHERE a.auction_number = 161
    GROUP BY a.auction_number
""")

row = c2.fetchone()
if row:
    auction, lots, min_id, max_id, last_time = row
    print(f"Auction:       {auction}")
    print(f"Lots scraped:  {lots:,}")
    print(f"ID range:      {min_id:,} to {max_id:,}")
    print(f"Last scraped:  {last_time}")
    
    expected_start = 775956
    expected_end   = 782840
    expected_range = expected_end - expected_start
    coverage = (max_id - min_id) / expected_range * 100
    print(f"\nExpected range:   {expected_start:,} to {expected_end:,}")
    print(f"Expected IDs:     {expected_range:,}")
    print(f"Coverage:         {coverage:.1f}%")
else:
    print("No data for auction 161 yet")

conn2.close()

Auction:       161
Lots scraped:  2,792
ID range:      775,956 to 778,925
Last scraped:  2026-03-25 08:33:24

Expected range:   775,956 to 782,840
Expected IDs:     6,884
Coverage:         43.1%


In [51]:
def extract_year(date_str):
    """
    Extract a 4-digit year from any date string format.
    Returns an integer year or None if nothing found.
    """
    if not date_str or pd.isna(date_str):
        return None
    
    date_str = str(date_str).strip()
    
    four_digit = re.findall(r'\b(19\d{2}|20\d{2})\b', date_str)
    if four_digit:
        years = [int(y) for y in four_digit]
        return min(years)
    
    return None

test_cases = [
    "1991", "June 2008", "02.12.2015", "10.2012",
    "25.07.1978", "1974 & 1975", "circa 1970",
    "Spring 1994", "19.03.1993", None, "unknown"
]

print("Date parser test:")
print("-" * 35)
for test in test_cases:
    result = extract_year(test)
    print(f"  {str(test):20s}  →  {result}")

Date parser test:
-----------------------------------
  1991                  →  1991
  June 2008             →  2008
  02.12.2015            →  2015
  10.2012               →  2012
  25.07.1978            →  1978
  1974 & 1975           →  1974
  circa 1970            →  1970
  Spring 1994           →  1994
  19.03.1993            →  1993
  None                  →  None
  unknown               →  None


In [52]:
def extract_year(date_str):
    """
    Extract a 4-digit year from any date string format.
    Returns (year, is_approximate) tuple.
    is_approximate only True when year is not None.
    """
    if not date_str or pd.isna(date_str):
        return None, False
    
    date_str = str(date_str).strip().lower()
    
    is_approximate = any(word in date_str for word in 
                        ["circa", "around", "approx", "c.", "ca."])
    
    four_digit = re.findall(r'\b(19\d{2}|20\d{2})\b', date_str)
    
    if not four_digit:
        return None, False
    
    years = [int(y) for y in four_digit]
    
    if len(years) > 1:
        year_range = max(years) - min(years)
        if year_range <= 3:
            return min(years), True
        else:
            return None, False
    
    return years[0], is_approximate

test_cases = [
    "1991", "June 2008", "02.12.2015",
    "1974 & 1975", "1968/1969", "1960 & 1965",
    "circa 1970", "Spring 1994", "19.03.1993",
    None, "unknown", "c. 1965", "around 1980",
]

print("Date parser test:")
print("-" * 50)
for test in test_cases:
    year, approx = extract_year(test)
    flag = " (approximate)" if approx else ""
    print(f"  {str(test):22s}  →  {str(year):6}{flag}")

Date parser test:
--------------------------------------------------
  1991                    →  1991  
  June 2008               →  2008  
  02.12.2015              →  2015  
  1974 & 1975             →  1974   (approximate)
  1968/1969               →  1968   (approximate)
  1960 & 1965             →  None  
  circa 1970              →  1970   (approximate)
  Spring 1994             →  1994  
  19.03.1993              →  1993  
  None                    →  None  
  unknown                 →  None  
  c. 1965                 →  1965   (approximate)
  around 1980             →  1980   (approximate)
